## 1. Database Setup with Document Chunks, Embeddings, and Metadata

In [ ]:
import sqlite3
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample documents
documents = [
    {"doc_id": 1, "title": "Machine Learning Basics", "source": "ml_guide.pdf", 
     "text": "Machine learning is a subset of artificial intelligence that focuses on data and algorithms. It enables computers to learn from experience without being explicitly programmed."},
    {"doc_id": 2, "title": "Deep Learning Introduction", "source": "dl_intro.pdf",
     "text": "Deep learning uses neural networks with multiple layers. These networks can learn complex patterns in large amounts of data."},
    {"doc_id": 3, "title": "Natural Language Processing", "source": "nlp_basics.pdf",
     "text": "NLP enables computers to understand and process human language. It combines linguistics with machine learning."},
    {"doc_id": 4, "title": "Computer Vision", "source": "cv_fundamentals.pdf",
     "text": "Computer vision allows machines to interpret visual information from the world. It uses deep learning for image recognition."},
    {"doc_id": 5, "title": "Reinforcement Learning", "source": "rl_guide.pdf",
     "text": "Reinforcement learning trains agents through rewards and penalties. The agent learns optimal actions through trial and error."},
    {"doc_id": 6, "title": "Data Preprocessing", "source": "data_prep.pdf",
     "text": "Data preprocessing involves cleaning and transforming raw data. This step is crucial for successful machine learning."},
    {"doc_id": 7, "title": "Neural Networks", "source": "nn_basics.pdf",
     "text": "Neural networks are inspired by biological brains. They consist of interconnected nodes that process information."},
    {"doc_id": 8, "title": "Supervised Learning", "source": "supervised_ml.pdf",
     "text": "Supervised learning uses labeled training data. The model learns to map inputs to known outputs."},
    {"doc_id": 9, "title": "Unsupervised Learning", "source": "unsupervised_ml.pdf",
     "text": "Unsupervised learning finds patterns in unlabeled data. Common techniques include clustering and dimensionality reduction."},
    {"doc_id": 10, "title": "Model Evaluation", "source": "eval_metrics.pdf",
     "text": "Model evaluation measures how well algorithms perform. Common metrics include accuracy, precision, and recall."}
]

# Create chunks (in this case, each document is one chunk)
chunks = [(doc["doc_id"], doc["text"]) for doc in documents]

def build_database():
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    
    # Create documents table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS documents (
            doc_id INTEGER PRIMARY KEY,
            title TEXT,
            source TEXT
        )
    ''')
    
    # Create chunks table with metadata
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS chunks (
            chunk_id INTEGER PRIMARY KEY AUTOINCREMENT,
            doc_id INTEGER,
            chunk_text TEXT,
            faiss_idx INTEGER,
            FOREIGN KEY (doc_id) REFERENCES documents(doc_id)
        )
    ''')
    
    # Create FTS5 table for full-text search
    cursor.execute('''
        CREATE VIRTUAL TABLE IF NOT EXISTS doc_chunks_fts
        USING fts5(chunk_text, content='chunks', content_rowid='chunk_id')
    ''')
    
    # Insert documents
    for doc in documents:
        cursor.execute('INSERT OR REPLACE INTO documents (doc_id, title, source) VALUES (?, ?, ?)',
                      (doc["doc_id"], doc["title"], doc["source"]))
    
    # Insert chunks
    for idx, (doc_id, text) in enumerate(chunks):
        cursor.execute('INSERT INTO chunks (doc_id, chunk_text, faiss_idx) VALUES (?, ?, ?)',
                      (doc_id, text, idx))
    
    # Populate FTS5 table
    cursor.execute('INSERT INTO doc_chunks_fts(doc_chunks_fts) VALUES(\'rebuild\')')
    
    conn.commit()
    conn.close()
    print(f"✅ Database created with {len(documents)} documents and {len(chunks)} chunks")

def build_faiss_index():
    chunk_texts = [text for _, text in chunks]
    embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings.astype('float32'))
    
    faiss.write_index(index, 'faiss_index.bin')
    print(f"✅ FAISS index created with {index.ntotal} vectors (dimension={dimension})")
    return index

# Build database and index
build_database()
faiss_index = build_faiss_index()

## 2. Hybrid Retrieval Pipeline

In [ ]:
def vector_search(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    distances, indices = faiss_index.search(query_embedding.astype('float32'), top_k)
    
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        cursor.execute('''
            SELECT c.chunk_id, c.chunk_text, d.title, d.source
            FROM chunks c
            JOIN documents d ON c.doc_id = d.doc_id
            WHERE c.faiss_idx = ?
        ''', (int(idx),))
        row = cursor.fetchone()
        if row:
            results.append({
                "chunk_id": row[0],
                "text": row[1],
                "title": row[2],
                "source": row[3],
                "score": float(1 / (1 + dist))
            })
    
    conn.close()
    return results

def fts5_search(query, top_k=3):
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT c.chunk_id, c.chunk_text, d.title, d.source, rank
        FROM doc_chunks_fts fts
        JOIN chunks c ON fts.rowid = c.chunk_id
        JOIN documents d ON c.doc_id = d.doc_id
        WHERE doc_chunks_fts MATCH ?
        ORDER BY rank
        LIMIT ?
    ''', (query, top_k))
    
    results = []
    for row in cursor.fetchall():
        results.append({
            "chunk_id": row[0],
            "text": row[1],
            "title": row[2],
            "source": row[3],
            "score": float(-row[4])
        })
    
    conn.close()
    return results

def bm25_search(query, top_k=3):
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    cursor.execute('SELECT chunk_id, chunk_text FROM chunks')
    all_chunks = cursor.fetchall()
    
    tokenized_corpus = [text.lower().split() for _, text in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk_id = all_chunks[idx][0]
        cursor.execute('''
            SELECT c.chunk_text, d.title, d.source
            FROM chunks c
            JOIN documents d ON c.doc_id = d.doc_id
            WHERE c.chunk_id = ?
        ''', (chunk_id,))
        row = cursor.fetchone()
        if row:
            results.append({
                "chunk_id": chunk_id,
                "text": row[0],
                "title": row[1],
                "source": row[2],
                "score": float(scores[idx])
            })
    
    conn.close()
    return results

def hybrid_search_weighted(query, alpha=0.5, top_k=3):
    vector_results = vector_search(query, top_k=10)
    bm25_results = bm25_search(query, top_k=10)
    
    combined_scores = {}
    
    for result in vector_results:
        chunk_id = result["chunk_id"]
        combined_scores[chunk_id] = {"info": result, "score": alpha * result["score"]}
    
    for result in bm25_results:
        chunk_id = result["chunk_id"]
        if chunk_id in combined_scores:
            combined_scores[chunk_id]["score"] += (1 - alpha) * result["score"]
        else:
            combined_scores[chunk_id] = {"info": result, "score": (1 - alpha) * result["score"]}
    
    sorted_results = sorted(combined_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_k]
    return [{**item["info"], "score": item["score"]} for _, item in sorted_results]

def hybrid_search_rrf(query, k=60, top_k=3):
    vector_results = vector_search(query, top_k=10)
    bm25_results = bm25_search(query, top_k=10)
    
    rrf_scores = {}
    
    for rank, result in enumerate(vector_results, 1):
        chunk_id = result["chunk_id"]
        rrf_scores[chunk_id] = {"info": result, "score": 1 / (k + rank)}
    
    for rank, result in enumerate(bm25_results, 1):
        chunk_id = result["chunk_id"]
        if chunk_id in rrf_scores:
            rrf_scores[chunk_id]["score"] += 1 / (k + rank)
        else:
            rrf_scores[chunk_id] = {"info": result, "score": 1 / (k + rank)}
    
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_k]
    return [{**item["info"], "score": item["score"]} for _, item in sorted_results]

print("✅ Hybrid retrieval pipeline loaded")

## 3. Evaluation with 10+ Queries and Metrics

In [ ]:
import json

# Test queries with ground truth (relevant doc_ids)
test_queries = [
    {"query": "What is machine learning?", "relevant_docs": [1, 6, 8]},
    {"query": "deep learning neural networks", "relevant_docs": [2, 7]},
    {"query": "natural language processing", "relevant_docs": [3]},
    {"query": "computer vision image recognition", "relevant_docs": [4]},
    {"query": "reinforcement learning rewards", "relevant_docs": [5]},
    {"query": "data cleaning preprocessing", "relevant_docs": [6]},
    {"query": "supervised learning labeled data", "relevant_docs": [8]},
    {"query": "unsupervised learning clustering", "relevant_docs": [9]},
    {"query": "model accuracy metrics", "relevant_docs": [10]},
    {"query": "artificial intelligence algorithms", "relevant_docs": [1, 2]},
    {"query": "neural network layers", "relevant_docs": [2, 7]},
    {"query": "training data machine learning", "relevant_docs": [1, 6, 8]}
]

def get_doc_ids_from_results(results):
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    doc_ids = []
    for result in results:
        cursor.execute('SELECT doc_id FROM chunks WHERE chunk_id = ?', (result["chunk_id"],))
        row = cursor.fetchone()
        if row:
            doc_ids.append(row[0])
    conn.close()
    return doc_ids

def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    retrieved_set = set(retrieved_docs[:k])
    relevant_set = set(relevant_docs)
    if not relevant_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(relevant_set)

def calculate_hit_rate_at_k(retrieved_docs, relevant_docs, k=3):
    retrieved_set = set(retrieved_docs[:k])
    relevant_set = set(relevant_docs)
    return 1.0 if retrieved_set & relevant_set else 0.0

def evaluate_method(method_name, search_function):
    total_recall = 0
    total_hit_rate = 0
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {method_name}")
    print(f"{'='*60}")
    
    for test in test_queries:
        query = test["query"]
        relevant_docs = test["relevant_docs"]
        
        results = search_function(query)
        retrieved_docs = get_doc_ids_from_results(results)
        
        recall = calculate_recall_at_k(retrieved_docs, relevant_docs)
        hit_rate = calculate_hit_rate_at_k(retrieved_docs, relevant_docs)
        
        total_recall += recall
        total_hit_rate += hit_rate
        
        print(f"Query: {query[:40]}...")
        print(f"  Retrieved: {retrieved_docs[:3]} | Relevant: {relevant_docs}")
        print(f"  Recall@3: {recall:.2f} | Hit@3: {hit_rate:.2f}")
    
    avg_recall = total_recall / len(test_queries)
    avg_hit_rate = total_hit_rate / len(test_queries)
    
    print(f"\n{method_name} Results:")
    print(f"  Average Recall@3: {avg_recall:.2%}")
    print(f"  Average Hit Rate@3: {avg_hit_rate:.2%}")
    
    return {"recall@3": avg_recall, "hit_rate@3": avg_hit_rate}

# Run evaluations
results = {}
results["vector_only"] = evaluate_method("Vector Search (FAISS)", vector_search)
results["keyword_only_bm25"] = evaluate_method("Keyword Search (BM25)", bm25_search)
results["keyword_only_fts5"] = evaluate_method("Keyword Search (FTS5)", fts5_search)
results["hybrid_weighted"] = evaluate_method("Hybrid (Weighted α=0.5)", hybrid_search_weighted)
results["hybrid_rrf"] = evaluate_method("Hybrid (RRF)", hybrid_search_rrf)

# Save results
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Evaluation complete. Results saved to evaluation_results.json")

## 4. FastAPI Endpoint Implementation

In [ ]:
%%writefile hybrid_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import sqlite3
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from typing import List, Dict

app = FastAPI(title="Hybrid Search API")

# Load models and index at startup
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index = faiss.read_index('faiss_index.bin')

class SearchQuery(BaseModel):
    query: str
    top_k: int = 3
    alpha: float = 0.5

class SearchResult(BaseModel):
    chunk_id: int
    text: str
    title: str
    source: str
    score: float

def vector_search(query: str, top_k: int = 3) -> List[Dict]:
    query_embedding = embedding_model.encode([query])
    distances, indices = faiss_index.search(query_embedding.astype('float32'), top_k)
    
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        cursor.execute('''
            SELECT c.chunk_id, c.chunk_text, d.title, d.source
            FROM chunks c
            JOIN documents d ON c.doc_id = d.doc_id
            WHERE c.faiss_idx = ?
        ''', (int(idx),))
        row = cursor.fetchone()
        if row:
            results.append({
                "chunk_id": row[0],
                "text": row[1],
                "title": row[2],
                "source": row[3],
                "score": float(1 / (1 + dist))
            })
    
    conn.close()
    return results

def bm25_search(query: str, top_k: int = 3) -> List[Dict]:
    conn = sqlite3.connect('hybrid_search.db')
    cursor = conn.cursor()
    cursor.execute('SELECT chunk_id, chunk_text FROM chunks')
    all_chunks = cursor.fetchall()
    
    tokenized_corpus = [text.lower().split() for _, text in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk_id = all_chunks[idx][0]
        cursor.execute('''
            SELECT c.chunk_text, d.title, d.source
            FROM chunks c
            JOIN documents d ON c.doc_id = d.doc_id
            WHERE c.chunk_id = ?
        ''', (chunk_id,))
        row = cursor.fetchone()
        if row:
            results.append({
                "chunk_id": chunk_id,
                "text": row[0],
                "title": row[1],
                "source": row[2],
                "score": float(scores[idx])
            })
    
    conn.close()
    return results

@app.post("/hybrid_search", response_model=List[SearchResult])
async def hybrid_search(query: SearchQuery):
    try:
        vector_results = vector_search(query.query, top_k=10)
        bm25_results = bm25_search(query.query, top_k=10)
        
        combined_scores = {}
        
        for result in vector_results:
            chunk_id = result["chunk_id"]
            combined_scores[chunk_id] = {"info": result, "score": query.alpha * result["score"]}
        
        for result in bm25_results:
            chunk_id = result["chunk_id"]
            if chunk_id in combined_scores:
                combined_scores[chunk_id]["score"] += (1 - query.alpha) * result["score"]
            else:
                combined_scores[chunk_id] = {"info": result, "score": (1 - query.alpha) * result["score"]}
        
        sorted_results = sorted(combined_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:query.top_k]
        return [{**item["info"], "score": item["score"]} for _, item in sorted_results]
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "Hybrid Search API", "endpoint": "/hybrid_search"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

## Test the API

In [ ]:
# Run this in a separate terminal:
# uvicorn hybrid_api:app --reload

# Test with curl:
# curl -X POST "http://localhost:8000/hybrid_search" \
#      -H "Content-Type: application/json" \
#      -d '{"query": "machine learning", "top_k": 3, "alpha": 0.5}'

print("API endpoint created: hybrid_api.py")
print("Run: uvicorn hybrid_api:app --reload")
print("Test at: http://localhost:8000/docs")

## Summary

### Deliverables Completed:

1. **✅ SQLite+FAISS Index**
   - Database: `hybrid_search.db` with documents, chunks, and FTS5 index
   - FAISS index: `faiss_index.bin` with embeddings

2. **✅ Hybrid Retrieval Pipeline**
   - FAISS vector search
   - BM25 keyword search
   - FTS5 full-text search
   - Weighted score merging (alpha parameter)
   - Reciprocal Rank Fusion (RRF)

3. **✅ Evaluation with 12 Queries**
   - Recall@3 and Hit Rate@3 metrics
   - Comparison of vector-only, keyword-only, and hybrid approaches
   - Results saved to `evaluation_results.json`

4. **✅ FastAPI Endpoint**
   - `/hybrid_search` endpoint returning top-3 results as JSON
   - Configurable alpha and top_k parameters
   - Interactive docs at `/docs`